# Grade Performance Analysis
**Joseph Moore**  
CPSC 222 — Introduction to Data Science, Spring 2026  
Gonzaga University

---
## Introduction

### Domain
As a student, my grade performance is something I interact with constantly but rarely analyze systematically. I wanted to know whether patterns in *how* and *when* I submit assignments actually correlate with my scores — for instance, do I score better on assignments due earlier in the week, or does submitting early versus on-time make a difference? Hopefully the gained insight can help me improve my personal study habits as well as better understand how I perform on differing types of assignemnts. 

### Dataset Format
The dataset consists of two CSV files:
- **`projectData.csv`** — raw grade export from Canvas, manually formatted. Each row is one assignment.
- **`calendar.csv`** — a reference table mapping days of the week to a weekend flag (`IsWeekend`). This table is joined with the grade data to add context about due-date timing.

### Tables and Attributes
**`projectData.csv`** (~92 rows, one per assignment):
| Column | Type | Description |
|---|---|---|
| Name | string | Assignment name |
| Category | string | Raw Canvas category label |
| Due | string | Due date/time (e.g. `Apr 14 11:59pm`) |
| Submitted | string | Submission timestamp |
| Score | float | Points earned |
| OutOf | float | Maximum possible points |

**`calendar.csv`** (7 rows, one per day of week):
| Column | Type | Description |
|---|---|---|
| DayOfWeek | string | Day name (e.g. `Monday`) |
| IsWeekend | int | 1 if Saturday or Sunday, 0 otherwise |

Data was collected across two Spring 2026 courses: UI/UX Design and Data Science.

### Classification Task
The goal is to predict whether a given assignment will result in a **high score (≥ 85%)** using features like assignment category, day of the week it was due, how early it was submitted, and its point value. This is a binary classification problem: `HighScore = 1` or `HighScore = 0`.

### Potential Impacts and Stakeholders
- **Students** could use patterns like these to prioritize time management or identify which assignment types need more attention.
- **Instructors** might find it useful to understand whether due-date timing affects student performance.
- Ethically, applying a model like this to make real decisions about students would be ill advised given the small dataset and high risk of overfitting.

---
## Data Preparation

### Load and Clean Grade Data
The raw grade CSV is loaded and cleaned. Numeric columns (`Score`, `OutOf`) are coerced to handle any non-numeric entries. A `Percent` column is calculated, and a standardized `CleanCategory` column normalizes the varied category labels into four groups: *quizzes*, *exams/tests*, *small assignments*, and *big assignments*.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')

from utilsProjectData import load_and_clean_grades, add_timing_columns

df = load_and_clean_grades('projectData.csv')
df = add_timing_columns(df)

print(f"Rows: {len(df)}, Columns: {list(df.columns)}")
df.head()

Rows: 92, Columns: ['Name', 'Category', 'Due', 'Submitted', 'Score', 'OutOf', 'CleanCategory', 'Percent', 'DueDate', 'SubDate', 'DayOfWeek', 'DaysEarly']


,Name,Category,Due,Submitted,Score,OutOf,CleanCategory,Percent,DueDate,SubDate,DayOfWeek,DaysEarly
0,MA3,Mini Assignments,Jan 28 11:59pm,NaN,4.0,5.0,small assignments,80.0,2026-01-28 23:59:00,NaT,Wednesday,NaN
1,DA1,Data Assignments,Feb 1 11:59pm,Feb 1 5:58pm,46.0,50.0,big assignments,92.0,2026-02-01 23:59:00,2026-02-01 17:58:00,Sunday,0.0
2,MA4 - Handout,Mini Assignments,Feb 2 11:59pm,NaN,0.0,5.0,small assignments,0.0,2026-02-02 23:59:00,NaT,Monday,NaN
3,MA4,Mini Assignments,Feb 4 11:59pm,Feb 4 10:45am,5.0,5.0,small assignments,100.0,2026-02-04 23:59:00,2026-02-04 10:45:00,Wednesday,0.0
4,DA2,Data Assignments,Feb 10 11:59pm,Feb 10 5:58pm,72.0,75.0,big assignments,96.0,2026-02-10 23:59:00,2026-02-10 17:58:00,Tuesday,0.0


The dataset came in fairly clean. The main cleaning steps were:
- Grade data was copy-pasted from Canvas and formatted into CSV with the help of ChatGPT, then manually reviewed for accuracy.
- A small number of rows have missing `Score` or `Submitted` values — these correspond to ungraded or unsubmitted assignments. They are dropped as needed during analysis using `dropna()`.
- Assignment categories were normalized from the Canvas labels into four groups via `clean_category()`.

### Join with Calendar Table
A calendar reference table (`calendar.csv`) maps each day of the week to a binary `IsWeekend` flag. This is joined onto the grade data on `DayOfWeek`, adding context about whether each assignment was due on a weekday or weekend.

In [2]:
from utilsProjectData import load_calendar, join_with_calendar

cal = load_calendar('calendar.csv')
print("Calendar table:")
print(cal)

df = join_with_calendar(df, cal)
print(f"\nAfter join — rows: {len(df)}, IsWeekend null count: {df['IsWeekend'].isna().sum()}")

KeyError: 'Date'

The merge is a left join on `DayOfWeek`, so all grade rows are preserved. Any assignment with a missing due date (and therefore no day of week) receives a null `IsWeekend` value, which is handled by `dropna()` downstream.

The final cleaned dataset is saved to CSV for reproducibility.

In [ ]:
df.to_csv('cleaned_grades.csv', index=False)
print("Saved cleaned_grades.csv")

---
## Exploratory Data Analysis

### Summary Statistics
Summary statistics (mean, median, std, min, max) are computed per assignment category. The mean and median help identify typical performance, while the standard deviation shows which categories have more variability.

In [ ]:
from utilsProjectData import summary_statistics

summary = summary_statistics(df)

Interpret the table above: categories with higher std values have more variable performance, while categories with high medians but lower means may include a few outlier low scores dragging the average down.

### Visualizations

#### Grade Distribution by Category
This boxplot shows the spread of percent scores for each assignment category. It helps identify categories with consistently high performance versus those with wider variance.

In [ ]:
from utilsProjectData import plot_grade_distribution_by_category

plot_grade_distribution_by_category(df)

#### Average Grade by Category
This bar chart gives a single summary value per category, making cross-category comparison easier than the boxplot alone.

In [ ]:
from utilsProjectData import plot_avg_grade_by_category

plot_avg_grade_by_category(df)

#### Average Grade by Day of Week
This chart explores whether assignments due on certain days correlate with higher or lower scores. With a small dataset, any pattern here is observational and may not generalize.

In [ ]:
from utilsProjectData import plot_avg_grade_by_day

plot_avg_grade_by_day(df)

#### Average Grade: Weekday vs Weekend Due Dates
Using the `IsWeekend` column from the calendar join, this chart compares average grades between assignments due on weekdays and those due on weekends.

In [ ]:
from utilsProjectData import plot_avg_grade_weekend_vs_weekday

plot_avg_grade_weekend_vs_weekday(df)

### Hypothesis Test 1 — Is My Average Grade Above 80%?

**H₀:** Mean grade ≤ 80%  
**H₁:** Mean grade > 80%  

A one-sample t-test is used to evaluate whether the average grade across all assignments is statistically significantly above 80%. A p-value below 0.05 leads to rejecting the null hypothesis.

In [ ]:
from utilsProjectData import test_mean_above_80

graded = df.dropna(subset=["Score", "OutOf"])
t_stat, p_value = test_mean_above_80(graded)

### Hypothesis Test 2 — Does Due Date (Weekday vs Weekend) Affect Grade?

**H₀:** Mean grade on weekday due dates = mean grade on weekend due dates  
**H₁:** Mean grade differs between weekday and weekend due dates  

A two-sample independent t-test is used. This leverages the calendar join: without the `IsWeekend` column, this test would not be possible directly from the grade data alone. A p-value below 0.05 leads to rejecting the null hypothesis.

In [ ]:
from utilsProjectData import test_weekday_vs_weekend

t_stat2, p_value2 = test_weekday_vs_weekend(df)

---
## Classification Results

### Setup
A binary target variable `HighScore` is created: 1 if the assignment percent is ≥ 85%, 0 otherwise. Four features are used:
- **CategoryEncoded** — numerically encoded assignment category
- **DayEncoded** — numerically encoded day of the week the assignment was due
- **DaysEarly** — days before the deadline the assignment was submitted (negative = late)
- **OutOf** — total point value of the assignment

The dataset is split 80/20 into training and test sets.

In [ ]:
from utilsProjectData import prepare_ml_data, print_class_distribution

X_train, X_test, y_train, y_test, features, ml_df = prepare_ml_data(df, threshold=85)

print_class_distribution(ml_df)
print(f"\nTraining set size: {len(X_train)}, Test set size: {len(X_test)}")

**Hypothesis:** Given the class imbalance (most assignments score ≥ 85%), both models are expected to predict the majority class in most cases, resulting in high accuracy but poor precision/recall for the minority (Low) class. The features available may not have enough signal to overcome this imbalance at this dataset size.

### KNN Classifier
A KNN classifier is trained across a range of k values (1–14). The plot below shows accuracy per k, helping select the best value rather than arbitrarily picking k=5.

In [ ]:
from utilsProjectData import run_knn

knn_model = run_knn(X_train, X_test, y_train, y_test, k_range=range(1, 15))

### Decision Tree Classifier
A decision tree with max depth of 4 is trained on the same features. The tree visualization shows which features the model splits on and at what thresholds.

In [ ]:
from utilsProjectData import run_decision_tree

dt_model = run_decision_tree(X_train, X_test, y_train, y_test, features)

### Comparison
Both models achieve similar overall accuracy, but this is misleading due to class imbalance — with the majority of test samples being `HighScore = 1`, a model that simply always predicts High will score well on accuracy while learning nothing. The classification report's precision and recall for class 0 (Low) are the more honest indicators of model usefulness. Neither model is expected to meaningfully distinguish Low from High grades at this sample size. The decision tree is slightly more interpretable because its splits can be read directly; the KNN is a black box by comparison.

---
## Conclusion

**Dataset limitation was the central challenge.** With only ~92 assignments across two courses, and far fewer rows after filtering for complete date and grade data, both classifiers were working with a very small test set. High accuracy figures are misleading — they reflect class imbalance (most grades ≥ 85%) rather than learned patterns.

**The hypothesis tests gave cleaner results.** The one-sample t-test confirmed whether overall performance is significantly above 80%, and the two-sample t-test (made possible by the calendar join) tested whether weekday vs. weekend deadlines correlate with grade differences. These are interpretable, statistically grounded conclusions regardless of dataset size.

**Potential improvements:**
- Collect data across more semesters and courses to increase n.
- Lower the `HighScore` threshold to 70% to reduce class imbalance.
- Add features like time of day submitted, or course identifier.

**Ethical impacts:** The stakeholders are primarily students and instructors. If a tool like this were deployed to make real decisions (e.g., flagging students for intervention), the risk of false positives on a skewed model would be high. Personal grade patterns also don't generalize across students, so any model trained on one person's data should never be used to make inferences about others. The dataset itself is private by nature and should not be shared publicly.

---
## Sources

- Grade data: personal Canvas export, Spring 2026 (UI/UX Design, CPSC 222)
- Calendar reference table: manually created
- ChatGPT (OpenAI): used to assist in formatting the raw Canvas copy-paste into CSV structure; all analysis code is original
- scikit-learn documentation: https://scikit-learn.org/stable/
- pandas documentation: https://pandas.pydata.org/docs/
- CPSC 222 course materials, Gonzaga University (Gina Sprint / Dominic MacIsaac)